# TAC-LAnoBERT v2 Phase 2: Projection Head Training

**Purpose**: Train dedicated projection head for early anomaly detection

---

## 🎯 Phase 2 Goals

### What is Projection Head?

A **learnable transformation** (768→256→64) that maps BERT embeddings to a specialized space optimized for:
- 🎯 **Early detection** - detect anomalies with longer lead time
- 📉 **FP reduction** - better separation between normal/anomaly
- 🧠 **Distance quality** - KNN works better in learned space

### Architecture

```
BERT [CLS] (768) → FC1 (256) → GELU → Dropout → FC2 (64)
                                                     ↓
                                              Memory Queue
                                              KNN Distance
```

### Training Strategy

**Contrastive Learning** with early detection objective:
- Pull normal sequences close together
- Push anomalies far from normals
- Maximize lead time between detection and failure

---

## 📊 Expected Improvements

| Metric | Phase 1 (KNN+PCA) | Phase 2 Target | Δ      |
|--------|-------------------|----------------|--------|
| F1     | ~0.985            | ≥ 0.990        | +0.5%  |
| FP     | ~1,000            | ≤ 500          | -50%   |
| EWR    | ~30%              | ≥ 40%          | +33%   |
| DLT    | ~450s             | ≥ 600s         | +33%   |

**Why better than PCA?**
- PCA: Linear, unsupervised, generic
- Projection Head: Non-linear, supervised, task-specific

---

## ⚙️ Configuration

**Config**: `configs/bgl_tac_v2_phase2.yaml`

**Prerequisites**:
- ✅ Phase 1 completed (optimized model validated)
- ✅ Pre-trained BERT (10 epochs)
- ✅ Preprocessed BGL data

**Training**:
- Freeze BERT encoder
- Train only projection head (2-3 epochs)
- Contrastive + Early Detection loss

---

## 🕐 Runtime

- **GPU (T4)**: ~1-2 hours (projection head only)
- **CPU**: ~4-6 hours

---

## 1. Setup Environment

In [ ]:
# Clone repository (if on Kaggle/Colab)
import os

if not os.path.exists('TAC-LAnoBERT-y'):
    print("📦 Cloning repository...")
    !git clone https://github.com/rubyhcm/TAC-LAnoBERT-y.git
    %cd TAC-LAnoBERT-y
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")
    if not os.getcwd().endswith('TAC-LAnoBERT-y'):
        %cd TAC-LAnoBERT-y
        print(f"📂 Changed to: {os.getcwd()}")

In [ ]:
# Install dependencies
print("📦 Installing dependencies...")
!pip install -r requirements.txt -q
print("✅ Dependencies installed")

In [ ]:
# Verify environment
import torch
import transformers
import numpy as np
import sys
from pathlib import Path

print("=" * 70)
print("ENVIRONMENT INFO")
print("=" * 70)
print(f"\nPython:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"\nCUDA:         {'✅ Available' if torch.cuda.is_available() else '❌ Not available (CPU mode)'}")
if torch.cuda.is_available():
    print(f"  GPU:        {torch.cuda.get_device_name(0)}")
    print(f"  Version:    {torch.version.cuda}")
    print(f"  Memory:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("  ⚠️  Running on CPU (training will be slower)")

print(f"\nWorking dir:  {os.getcwd()}")
print("\n" + "=" * 70)
print("✅ Environment ready")
print("=" * 70)

## 2. Check Prerequisites

Verify Phase 1 results and required data.

In [ ]:
# Check Phase 1 completion
import glob
import shutil

print("=" * 70)
print("CHECKING PREREQUISITES")
print("=" * 70)

def copy_from_kaggle_input(pattern, target_dir, name):
    """Search for and copy data from Kaggle input"""
    found = glob.glob(pattern, recursive=True)
    if found:
        src = found[0]
        if os.path.isdir(src):
            dst = os.path.join(target_dir, os.path.basename(src))
            if not os.path.exists(dst):
                print(f"\n📦 {name}: Found in input, copying...")
                print(f"   {src} → {dst}")
                os.makedirs(target_dir, exist_ok=True)
                shutil.copytree(src, dst)
                print("   ✅ Copied")
            else:
                print(f"\n✅ {name}: Already in working directory")
        return True
    return False

# 1. Trained BERT model (REQUIRED)
if not copy_from_kaggle_input(
    "/kaggle/input/**/BGL_tac_v2_2epochs",
    "outputs",
    "TAC v2 Model (10 epochs)"
):
    if os.path.exists("outputs/BGL_tac_v2_2epochs/model"):
        print("\n✅ TAC v2 Model: Already available")
    else:
        print("\n❌ TAC v2 Model: NOT FOUND!")
        print("   → Required for Phase 2 (frozen encoder)")

# 2. Phase 1 results (RECOMMENDED for comparison)
phase1_results = Path("outputs/BGL_tac_v2_optimized/results")
if not copy_from_kaggle_input(
    "/kaggle/input/**/BGL_tac_v2_optimized",
    "outputs",
    "Phase 1 Results"
):
    if phase1_results.exists():
        print("\n✅ Phase 1 Results: Already available")
    else:
        print("\n⚠️  Phase 1 Results: Not found (optional for comparison)")

# 3. BGL data (REQUIRED)
if not copy_from_kaggle_input(
    "/kaggle/input/**/BGL/BGL_test_parsed.log",
    "data",
    "BGL Data"
):
    if os.path.exists("data/BGL/BGL_test_parsed.log"):
        print("\n✅ BGL Data: Already available")
    else:
        print("\n❌ BGL Data: NOT FOUND!")

print("\n" + "=" * 70)

In [ ]:
# Verify critical files
print("=" * 70)
print("FILE VERIFICATION")
print("=" * 70)

critical_files = {
    "BERT Config": "outputs/BGL_tac_v2_2epochs/model/config.json",
    "BERT Weights": "outputs/BGL_tac_v2_2epochs/model/model.safetensors",
    "Time2Vec": "outputs/BGL_tac_v2_2epochs/model/time2vec.pt",
    "Tokenizer": "outputs/BGL_tac_v2_2epochs/tokenizer/BGL_LogBERT-vocab.txt",
    "Train Data": "data/BGL/BGL_train_normal_parsed.log",
    "Train Timestamps": "data/BGL/BGL_train_normal_parsed.timestamps",
    "Test Data": "data/BGL/BGL_test_parsed.log",
    "Test Labels": "data/BGL/BGL_test_label.log",
    "Test Timestamps": "data/BGL/BGL_test_parsed.timestamps",
}

all_ok = True
print("\nRequired files:")
for name, path in critical_files.items():
    # Check alternative paths
    if not os.path.exists(path) and "model/" in path:
        alt_path = path.replace("model/", "model/final/")
        if os.path.exists(alt_path):
            path = alt_path
    
    if os.path.exists(path):
        size = os.path.getsize(path)
        size_mb = size / (1024 * 1024)
        print(f"  ✅ {name:<20} ({size_mb:>8.2f} MB)")
    else:
        print(f"  ❌ {name:<20} NOT FOUND")
        print(f"     Expected: {path}")
        all_ok = False

print("\n" + "=" * 70)
if all_ok:
    print("✅ ALL FILES READY")
    print("=" * 70)
else:
    print("❌ MISSING FILES - Cannot proceed")
    print("=" * 70)
    raise FileNotFoundError("Required files missing. Run Phase 1 first or check Kaggle input.")

## 3. Create Phase 2 Configuration

Generate config for projection head training.

In [ ]:
# Create Phase 2 config (if not exists)
import yaml

phase2_config_path = "configs/bgl_tac_v2_phase2.yaml"

if os.path.exists(phase2_config_path):
    print(f"✅ Phase 2 config already exists: {phase2_config_path}")
else:
    print(f"📝 Creating Phase 2 config: {phase2_config_path}")
    
    # Load Phase 1 config as base
    with open("configs/bgl_tac_v2_optimized.yaml") as f:
        config = yaml.safe_load(f)
    
    # Update for Phase 2
    config['run_name'] = 'bgl_tac_v2_phase2'
    config['paths']['model_dir'] = 'outputs/BGL_tac_v2_phase2/model'
    config['paths']['result_dir'] = 'outputs/BGL_tac_v2_phase2/results'
    config['paths']['projection_head'] = 'outputs/BGL_tac_v2_phase2/projection_head.pt'
    
    # Enable projection head
    if 'tac_v2' not in config:
        config['tac_v2'] = {}
    
    config['tac_v2']['projection_head'] = {
        'enabled': True,
        'input_dim': 768,
        'hidden_dim': 256,
        'output_dim': 64,
        'dropout': 0.1,
        'pretrained_bert': 'outputs/BGL_tac_v2_2epochs/model',  # Frozen encoder
    }
    
    # Projection head training settings
    config['projection_training'] = {
        'num_epochs': 3,
        'batch_size': 64,
        'learning_rate': 1.0e-3,
        'weight_decay': 0.01,
        'warmup_ratio': 0.1,
        'freeze_bert': True,
        'loss_weights': {
            'contrastive': 1.0,
            'early_detection': 0.5,
            'separation': 0.3,
        },
        'contrastive': {
            'temperature': 0.07,
            'margin': 0.5,
        },
        'early_detection': {
            'lead_time_target': 600,  # 10 min
            'penalty_weight': 2.0,
        },
    }
    
    # Update memory queue to use projection head output
    config['tac']['memory']['pca_components'] = None  # Disable PCA when using projection
    
    # Save config
    os.makedirs('configs', exist_ok=True)
    with open(phase2_config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    
    print(f"✅ Config created")

# Load and display
with open(phase2_config_path) as f:
    config = yaml.safe_load(f)

print("\n" + "=" * 70)
print("PHASE 2 CONFIGURATION")
print("=" * 70)

print(f"\n📋 Basic Info:")
print(f"   Run name:    {config['run_name']}")
print(f"   Config:      {phase2_config_path}")

if 'projection_training' in config:
    pt = config['projection_training']
    print(f"\n🎯 Projection Head Training:")
    print(f"   Epochs:          {pt.get('num_epochs', 3)}")
    print(f"   Batch size:      {pt.get('batch_size', 64)}")
    print(f"   Learning rate:   {pt.get('learning_rate', 1e-3)}")
    print(f"   Freeze BERT:     {pt.get('freeze_bert', True)}")
    
    print(f"\n   Loss Weights:")
    for loss_name, weight in pt.get('loss_weights', {}).items():
        print(f"     • {loss_name:<20} {weight:.2f}")

proj = config.get('tac_v2', {}).get('projection_head', {})
if proj.get('enabled'):
    print(f"\n🏗️  Architecture:")
    print(f"   Input:       {proj.get('input_dim', 768)}")
    print(f"   Hidden:      {proj.get('hidden_dim', 256)}")
    print(f"   Output:      {proj.get('output_dim', 64)}")
    print(f"   Dropout:     {proj.get('dropout', 0.1)}")
    print(f"   Base model:  {proj.get('pretrained_bert', 'N/A')}")

print("\n" + "=" * 70)

## 4. Train Projection Head 🚀

Train the projection head with frozen BERT encoder.

**Training objectives**:
1. **Contrastive Loss**: Separate normal/anomaly clusters
2. **Early Detection Loss**: Maximize lead time
3. **Separation Loss**: Maximize inter-class distance

**BERT is frozen** - only projection head weights are updated.

In [ ]:
# Check if projection head already trained
proj_checkpoint = Path("outputs/BGL_tac_v2_phase2/projection_head.pt")

if proj_checkpoint.exists():
    print("✅ Projection head checkpoint already exists")
    print(f"   Path: {proj_checkpoint}")
    print(f"   Size: {proj_checkpoint.stat().st_size / (1024*1024):.2f} MB")
    print("\n   Remove checkpoint to retrain, or skip to inference")
    skip_training = True
else:
    print("📊 No existing checkpoint found")
    print(f"   Will train projection head and save to: {proj_checkpoint}")
    skip_training = False

In [ ]:
# Train projection head
import time

if not skip_training:
    print("=" * 70)
    print("TRAINING PROJECTION HEAD")
    print("=" * 70)
    print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("\nThis will take ~1-2 hours on GPU (T4)...")
    print("BERT encoder is frozen - only projection head trains\n")
    
    start_time = time.time()
    
    # Run projection head training
    # Note: This assumes you have train_projection_head.py script
    # If not implemented yet, this will be a placeholder
    !python -m tac_lanobert.train_projection_head --config configs/bgl_tac_v2_phase2.yaml
    
    end_time = time.time()
    duration = end_time - start_time
    hours = int(duration // 3600)
    minutes = int((duration % 3600) // 60)
    
    print("\n" + "=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)
    print(f"End time:  {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Duration:  {hours}h {minutes}m")
    
    if proj_checkpoint.exists():
        print(f"\n✅ Checkpoint saved: {proj_checkpoint}")
        print(f"   Size: {proj_checkpoint.stat().st_size / (1024*1024):.2f} MB")
    else:
        print("\n⚠️  Checkpoint not found - check training logs")
    
    print("=" * 70)
else:
    print("\nℹ️  Using existing checkpoint (training skipped)")

## 5. Run Phase 2 Inference

Run inference with trained projection head.

In [ ]:
# Check if inference already done
results_dir = Path("outputs/BGL_tac_v2_phase2/results")
score_files = list(results_dir.glob("scores_*.npy")) if results_dir.exists() else []

if score_files:
    print("✅ Phase 2 inference results already exist")
    print(f"   Found {len(score_files)} score file(s) in {results_dir}")
    print("\n   Score files:")
    for sf in sorted(score_files):
        print(f"     • {sf.name}")
    print("\n   Skipping inference (remove results/ to re-run)")
    skip_inference = True
else:
    print("📊 No existing results found")
    print(f"   Will run inference and save to: {results_dir}")
    skip_inference = False

In [ ]:
# Run inference with projection head
if not skip_inference:
    print("=" * 70)
    print("STARTING PHASE 2 INFERENCE")
    print("=" * 70)
    print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("\nUsing trained projection head for embeddings...\n")
    
    start_time = time.time()
    
    # Run inference
    !python -m tac_lanobert.inference_tac --config configs/bgl_tac_v2_phase2.yaml
    
    end_time = time.time()
    duration = end_time - start_time
    minutes = int(duration // 60)
    seconds = int(duration % 60)
    
    print("\n" + "=" * 70)
    print("INFERENCE COMPLETE")
    print("=" * 70)
    print(f"End time:  {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Duration:  {minutes}m {seconds}s")
    print(f"Results:   {results_dir}")
    print("=" * 70)
else:
    print("\nℹ️  Using existing results (inference skipped)")

## 6. View Phase 2 Results

In [ ]:
# Load score files
results_dir = Path("outputs/BGL_tac_v2_phase2/results")
score_files = list(results_dir.glob("scores_*.npy"))

print("=" * 70)
print("PHASE 2 SCORE STATISTICS")
print("=" * 70)

if score_files:
    print(f"\nFound {len(score_files)} score file(s):\n")
    
    phase2_scores = {}
    for sf in sorted(score_files):
        scores = np.load(sf)
        name = sf.stem.replace('scores_', '')
        phase2_scores[name] = scores
        
        print(f"{name}:")
        print(f"  Lines:       {len(scores):,}")
        print(f"  Range:       [{scores.min():.6f}, {scores.max():.6f}]")
        print(f"  Mean ± Std:  {scores.mean():.6f} ± {scores.std():.6f}")
        print(f"  Median:      {np.median(scores):.6f}")
        print()
else:
    print("\n❌ No score files found!")
    print(f"   Expected in: {results_dir}")

print("=" * 70)

In [ ]:
# Parse metrics from report
import re
import json

def parse_report(report_path):
    """Parse TAC report file for metrics"""
    if not os.path.exists(report_path):
        return None
    
    with open(report_path, 'r') as f:
        content = f.read()
    
    metrics = {}
    
    # Basic metrics
    patterns = {
        'auroc': r'AUROC:\s+([0-9.e+-]+)',
        'f1': r'best_F1:\s+([0-9.e+-]+)',
        'precision': r'best_precision:\s+([0-9.e+-]+)',
        'recall': r'best_recall:\s+([0-9.e+-]+)',
        'threshold': r'best_threshold:\s+([0-9.e+-]+)',
    }
    
    for key, pattern in patterns.items():
        if m := re.search(pattern, content):
            metrics[key] = float(m.group(1))
    
    # Confusion matrix
    cm_pattern = r'confusion_matrix:.*?\[\[\s*(\d+)\s+(\d+)\s*\]\s*\[\s*(\d+)\s+(\d+)\s*\]\]'
    if m := re.search(cm_pattern, content, re.DOTALL):
        tn, fp, fn, tp = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        metrics.update({
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
            'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        })
    
    # Early detection metrics
    if m := re.search(r'DLT_mean:\s+([0-9.e+-]+)', content):
        metrics['dlt_mean'] = float(m.group(1))
    if m := re.search(r'EWR:\s+([0-9.e+-]+)', content):
        metrics['ewr'] = float(m.group(1))
    
    return metrics if metrics else None

# Find and parse Phase 2 report
report_files = list(results_dir.glob("*_report.txt"))

print("=" * 70)
print("PHASE 2 RESULTS")
print("=" * 70)

phase2_metrics = None
if report_files:
    phase2_metrics = parse_report(str(report_files[0]))
    
    if phase2_metrics:
        print(f"\n📊 Performance Metrics:\n")
        print(f"  F1-Score:    {phase2_metrics.get('f1', 0):.6f}")
        print(f"  Precision:   {phase2_metrics.get('precision', 0):.6f}")
        print(f"  Recall:      {phase2_metrics.get('recall', 0):.6f}")
        print(f"  AUROC:       {phase2_metrics.get('auroc', 0):.6f}")
        print(f"  FPR:         {phase2_metrics.get('fpr', 0)*100:.4f}%")
        
        print(f"\n🎯 Detection Metrics:\n")
        print(f"  Threshold:   {phase2_metrics.get('threshold', 0):.6f}")
        print(f"  True Pos:    {phase2_metrics.get('tp', 0):>8,}")
        print(f"  False Pos:   {phase2_metrics.get('fp', 0):>8,}")
        print(f"  True Neg:    {phase2_metrics.get('tn', 0):>8,}")
        print(f"  False Neg:   {phase2_metrics.get('fn', 0):>8,}")
        
        if 'dlt_mean' in phase2_metrics:
            print(f"\n⏰ Early Detection:\n")
            print(f"  DLT (mean):  {phase2_metrics['dlt_mean']:.1f} seconds")
            print(f"             ({phase2_metrics['dlt_mean']/60:.1f} minutes)")
            if 'ewr' in phase2_metrics:
                print(f"  EWR:         {phase2_metrics['ewr']*100:.2f}%")
        
        # Check Phase 2 targets
        print(f"\n✅ Phase 2 Target Achievement:\n")
        targets = [
            ('F1 ≥ 0.990', phase2_metrics.get('f1', 0) >= 0.990),
            ('FP ≤ 500', phase2_metrics.get('fp', 999999) <= 500),
            ('EWR ≥ 40%', phase2_metrics.get('ewr', 0) >= 0.40),
            ('DLT ≥ 600s', phase2_metrics.get('dlt_mean', 0) >= 600),
        ]
        for target_name, achieved in targets:
            icon = "✅" if achieved else "❌"
            print(f"  {icon} {target_name}")
    else:
        print("\n⚠️  Could not parse report file")
else:
    print("\n⚠️  No report files found")

print("\n" + "=" * 70)

## 7. Phase 1 vs Phase 2 Comparison

Compare PCA (Phase 1) vs Projection Head (Phase 2).

In [ ]:
# Load Phase 1 results for comparison
phase1_report = Path("outputs/BGL_tac_v2_optimized/results").glob("*_report.txt")
phase1_metrics = None

for report in phase1_report:
    phase1_metrics = parse_report(str(report))
    if phase1_metrics:
        break

if phase1_metrics and phase2_metrics:
    print("=" * 70)
    print("PHASE 1 (PCA) vs PHASE 2 (PROJECTION HEAD)")
    print("=" * 70)
    
    comparisons = [
        ('F1-Score', 'f1', 'higher', '.6f'),
        ('Precision', 'precision', 'higher', '.6f'),
        ('Recall', 'recall', 'higher', '.6f'),
        ('AUROC', 'auroc', 'higher', '.6f'),
        ('FPR', 'fpr', 'lower', '.6f'),
        ('False Positives', 'fp', 'lower', ',d'),
    ]
    
    print(f"\n{'Metric':<18} {'Phase 1 (PCA)':<16} {'Phase 2 (Proj)':<16} {'Δ':<12} {'Status'}")
    print("-" * 80)
    
    for metric_name, metric_key, better_direction, fmt in comparisons:
        p1_val = phase1_metrics.get(metric_key)
        p2_val = phase2_metrics.get(metric_key)
        
        if p1_val is not None and p2_val is not None:
            delta = p2_val - p1_val
            
            # Format values
            if metric_key in ['fpr']:
                p1_str = f"{p1_val*100:.4f}%"
                p2_str = f"{p2_val*100:.4f}%"
                delta_pct = (delta / p1_val * 100) if p1_val != 0 else 0
                delta_str = f"{delta_pct:+.2f}%"
            elif metric_key == 'fp':
                p1_str = f"{p1_val:,}"
                p2_str = f"{p2_val:,}"
                delta_str = f"{delta:+,}"
            else:
                p1_str = f"{p1_val:{fmt}}"
                p2_str = f"{p2_val:{fmt}}"
                delta_pct = (delta / p1_val * 100) if p1_val != 0 else 0
                delta_str = f"{delta_pct:+.2f}%"
            
            # Determine status
            if better_direction == 'lower':
                if delta < -0.001:  # Improved
                    status = "✅ Better"
                elif delta > 0.001:  # Worse
                    status = "❌ Worse"
                else:
                    status = "≈ Same"
            else:  # higher is better
                if delta > 0.001:
                    status = "✅ Better"
                elif delta < -0.001:
                    status = "❌ Worse"
                else:
                    status = "≈ Same"
            
            print(f"{metric_name:<18} {p1_str:<16} {p2_str:<16} {delta_str:<12} {status}")
    
    # Early detection comparison
    if 'dlt_mean' in phase1_metrics and 'dlt_mean' in phase2_metrics:
        print(f"\n⏰ Early Detection Comparison:\n")
        p1_dlt = phase1_metrics['dlt_mean']
        p2_dlt = phase2_metrics['dlt_mean']
        dlt_improve = ((p2_dlt - p1_dlt) / p1_dlt * 100) if p1_dlt > 0 else 0
        
        print(f"  DLT (Phase 1):   {p1_dlt:>8.1f}s ({p1_dlt/60:>5.1f} min)")
        print(f"  DLT (Phase 2):   {p2_dlt:>8.1f}s ({p2_dlt/60:>5.1f} min)")
        print(f"  Improvement:     {dlt_improve:>8.2f}%")
        
        if 'ewr' in phase1_metrics and 'ewr' in phase2_metrics:
            p1_ewr = phase1_metrics['ewr'] * 100
            p2_ewr = phase2_metrics['ewr'] * 100
            ewr_improve = p2_ewr - p1_ewr
            
            print(f"\n  EWR (Phase 1):   {p1_ewr:>8.2f}%")
            print(f"  EWR (Phase 2):   {p2_ewr:>8.2f}%")
            print(f"  Improvement:     {ewr_improve:>8.2f}pp")
    
    # Alert volume comparison
    if all(k in phase1_metrics for k in ['tp', 'fp']) and all(k in phase2_metrics for k in ['tp', 'fp']):
        p1_alerts = phase1_metrics['tp'] + phase1_metrics['fp']
        p2_alerts = phase2_metrics['tp'] + phase2_metrics['fp']
        alert_reduction = (p1_alerts - p2_alerts) / p1_alerts * 100 if p1_alerts > 0 else 0
        
        print(f"\n🔔 Alert Volume:\n")
        print(f"  Phase 1:       {p1_alerts:>8,} alerts")
        print(f"  Phase 2:       {p2_alerts:>8,} alerts")
        print(f"  Reduction:     {alert_reduction:>8.2f}%")
        
        if alert_reduction > 0:
            print(f"\n  ✅ Phase 2 reduces alert fatigue by {alert_reduction:.1f}%!")
    
    print("\n" + "=" * 70)
    
elif not phase1_metrics:
    print("\n⚠️  Phase 1 results not found for comparison")
    print("   Run Phase 1 notebook first for full comparison")
elif not phase2_metrics:
    print("\n⚠️  Phase 2 metrics not available")

## 8. Projection Head Analysis

Visualize what the projection head learned.

In [ ]:
# Load projection head and analyze
proj_checkpoint = Path("outputs/BGL_tac_v2_phase2/projection_head.pt")

if proj_checkpoint.exists():
    checkpoint = torch.load(proj_checkpoint, map_location='cpu')
    
    print("=" * 70)
    print("PROJECTION HEAD ANALYSIS")
    print("=" * 70)
    
    print("\n📦 Checkpoint Info:\n")
    print(f"  File size:     {proj_checkpoint.stat().st_size / (1024*1024):.2f} MB")
    
    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        print(f"  Layers:        {len(state_dict)} tensors")
        
        total_params = sum(p.numel() for p in state_dict.values())
        print(f"  Parameters:    {total_params:,}")
        
        print("\n  Layer shapes:")
        for name, tensor in state_dict.items():
            print(f"    {name:<30} {str(list(tensor.shape)):<20} ({tensor.numel():,} params)")
    
    if 'training_history' in checkpoint:
        history = checkpoint['training_history']
        print(f"\n📈 Training History:\n")
        
        if 'losses' in history:
            losses = history['losses']
            print(f"  Epochs:        {len(losses)}")
            print(f"  Final loss:    {losses[-1]:.6f}")
            print(f"  Best loss:     {min(losses):.6f}")
            
            if len(losses) > 1:
                improvement = (losses[0] - losses[-1]) / losses[0] * 100
                print(f"  Improvement:   {improvement:.2f}%")
    
    if 'config' in checkpoint:
        cfg = checkpoint['config']
        print(f"\n⚙️  Configuration:\n")
        for key, val in cfg.items():
            print(f"  {key:<20} {val}")
    
    print("\n" + "=" * 70)
else:
    print("⚠️  Projection head checkpoint not found")
    print(f"   Expected: {proj_checkpoint}")

## 9. Summary & Next Steps

In [ ]:
from datetime import datetime

print("=" * 70)
print("PHASE 2 SUMMARY")
print("=" * 70)

print(f"\n🎯 Configuration: {config['run_name']}")
print(f"   Config file:   configs/bgl_tac_v2_phase2.yaml")
print(f"   Results dir:   outputs/BGL_tac_v2_phase2/results")
print(f"   Completed:     {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if phase2_metrics:
    print(f"\n📊 Phase 2 Results:")
    print(f"   F1:        {phase2_metrics.get('f1', 0):.6f}")
    print(f"   Precision: {phase2_metrics.get('precision', 0):.6f}")
    print(f"   Recall:    {phase2_metrics.get('recall', 0):.6f}")
    print(f"   AUROC:     {phase2_metrics.get('auroc', 0):.6f}")
    print(f"   FPR:       {phase2_metrics.get('fpr', 0)*100:.4f}%")
    print(f"   FP Count:  {phase2_metrics.get('fp', 0):,}")
    
    if 'ewr' in phase2_metrics:
        print(f"   EWR:       {phase2_metrics['ewr']*100:.2f}%")
    if 'dlt_mean' in phase2_metrics:
        print(f"   DLT:       {phase2_metrics['dlt_mean']:.1f}s ({phase2_metrics['dlt_mean']/60:.1f} min)")

# Show improvements over Phase 1
if phase1_metrics and phase2_metrics:
    print(f"\n🚀 Phase 1 → Phase 2 Improvements:")
    
    improvements = []
    
    for metric_name, metric_key in [('F1', 'f1'), ('FP', 'fp'), ('EWR', 'ewr'), ('DLT', 'dlt_mean')]:
        if metric_key in phase1_metrics and metric_key in phase2_metrics:
            p1 = phase1_metrics[metric_key]
            p2 = phase2_metrics[metric_key]
            
            if metric_key == 'fp':
                delta = p2 - p1
                pct = (delta / p1 * 100) if p1 > 0 else 0
                improvements.append(f"   {metric_name}: {p1:,} → {p2:,} ({delta:+,}, {pct:+.1f}%)")
            elif metric_key == 'ewr':
                delta = (p2 - p1) * 100
                improvements.append(f"   {metric_name}: {p1*100:.2f}% → {p2*100:.2f}% ({delta:+.2f}pp)")
            elif metric_key == 'dlt_mean':
                delta = p2 - p1
                pct = (delta / p1 * 100) if p1 > 0 else 0
                improvements.append(f"   {metric_name}: {p1:.0f}s → {p2:.0f}s ({delta:+.0f}s, {pct:+.1f}%)")
            else:
                delta = p2 - p1
                pct = (delta / p1 * 100) if p1 > 0 else 0
                improvements.append(f"   {metric_name}: {p1:.6f} → {p2:.6f} ({pct:+.2f}%)")
    
    for imp in improvements:
        print(imp)

print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)

if phase2_metrics:
    # Check if Phase 2 targets met
    f1_ok = phase2_metrics.get('f1', 0) >= 0.990
    fp_ok = phase2_metrics.get('fp', 999999) <= 500
    ewr_ok = phase2_metrics.get('ewr', 0) >= 0.40
    dlt_ok = phase2_metrics.get('dlt_mean', 0) >= 600
    
    targets_met = sum([f1_ok, fp_ok, ewr_ok, dlt_ok])
    
    if targets_met >= 3:
        print(f"\n✅ Phase 2 targets achieved ({targets_met}/4)!")
        print("\n📋 Recommended actions:")
        print("   1. ✅ Document Phase 2 results and improvements")
        print("   2. 📊 Run ablation study (projection dims, loss weights)")
        print("   3. 🔬 Analyze failure cases and edge scenarios")
        print("   4. 🚀 Prepare production deployment")
        print("   5. 📝 Write research paper/technical report")
    else:
        print(f"\n⚠️  Phase 2 targets partially met ({targets_met}/4)")
        print("\n📋 Recommended actions:")
        print("   1. 🔍 Analyze what projection head learned")
        print("   2. ⚙️  Tune loss weights and architecture")
        print("   3. 📊 Try different projection dimensions")
        print("   4. 🔄 Consider fine-tuning BERT encoder")
else:
    print("\n⚠️  No metrics available - check inference logs")

print("\n" + "=" * 70)
print("✅ PHASE 2 NOTEBOOK COMPLETE")
print("=" * 70)

## 10. Export Results

In [ ]:
# Export Phase 2 summary
if phase2_metrics:
    summary = {
        'phase': 2,
        'config': config['run_name'],
        'timestamp': datetime.now().isoformat(),
        'projection_head': {
            'architecture': f"{proj.get('input_dim')}→{proj.get('hidden_dim')}→{proj.get('output_dim')}",
            'checkpoint': str(proj_checkpoint),
            'trained': proj_checkpoint.exists(),
        },
        'metrics': phase2_metrics,
        'comparison': {
            'phase1': phase1_metrics if phase1_metrics else None,
        },
    }
    
    output_json = results_dir / "phase2_summary.json"
    with open(output_json, 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    
    print(f"✅ Phase 2 summary exported to: {output_json}")
    print(f"\n   Use this for:")
    print(f"   • Comparison with future experiments")
    print(f"   • Research paper/report data")
    print(f"   • Production deployment reference")
else:
    print("⚠️  No metrics to export")